# OpenPlaque — Coronary CPR Roadmap + RCA PCAT Visualization

This notebook replaces slice/hotspot ranking with a **vessel-centered curved-planar-reformat (CPR) roadmap**.

Important: the Siemens curved coronary series are rotated whole-vessel reformats, **not anatomical cross-sectional stacks**. Therefore frame index is never interpreted as distance along the artery. For each LAD/RCA/LCX series, this notebook automatically selects the most informative CPR rotation, isolates the central coronary-vessel component in that frame, crops away reformat borders/text, and overlays only plaque near that central vessel.

Canonical TPV is unchanged. The plaque display filter is visualization-only.

RCA PCAT is shown separately in artery-centered source-CCTA cross-sections and a longitudinal 10–50 mm ribbon. Plaque CPR and PCAT source-volume views are **not spatially co-registered** and are not overlaid on each other.

Research use only. PCAT attenuation is an imaging surrogate related to perivascular inflammation; it is not a direct inflammation measurement and is not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL — mount Google Drive first.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch plaque-roadmap-visualization-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


In [ ]:
import os, sys, shutil, zipfile, base64
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.ndimage import map_coordinates
from IPython.display import display

REPO=Path('/content/OpenPlaque'); sys.path.insert(0,str(REPO/'src'))
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'Coronary_CPR_Roadmap_Report'; OUT.mkdir(parents=True,exist_ok=True)

os.environ['nnUNet_raw']='/content/nnUNet_raw'
os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'
os.environ['nnUNet_results']='/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'],os.environ['nnUNet_preprocessed'],os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True,exist_ok=True)

model_zip=ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
model_target=Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists(): raise FileNotFoundError(model_zip)
    with zipfile.ZipFile(model_zip) as z: z.extractall('/content/nnUNet_results')

drive_zip=ROOT/'Full_DICOM.zip'; local_zip=Path('/content/Full_DICOM.zip')
if not drive_zip.exists(): raise FileNotFoundError(drive_zip)
if not local_zip.exists() or local_zip.stat().st_size!=drive_zip.stat().st_size:
    shutil.copyfile(drive_zip,local_zip)

from openplaque.study import OpenPlaqueStudy
shutil.rmtree('/content/full_dicom_roadmap',ignore_errors=True)
study=OpenPlaqueStudy(str(local_zip),extract_root='/content/full_dicom_roadmap')
print('Output:',OUT)


## 1. Canonical plaque segmentation

Canonical refinement remains `min_component_voxels=10`, `lumen_distance_voxels=1`. Nothing in the visualization changes the quantitative TPV.


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series

fallback={'RCA':1035,'LCX':1039,'LAD':1043}
series_map,_=detect_artery_series(study,fallback_series=fallback,return_candidates=True)
reports=[]
for vessel in ['LAD','RCA','LCX']:
    image,volume,_=study.load_series(series_map[vessel])
    print('Segmenting',vessel,'series',series_map[vessel],volume.shape)
    reports.append(segment_vessel(image,volume,vessel))

def canonical_refine(r):
    return refine_plaque_mask(
        volume=r.volume, mask=r.mask, spacing=r.mask_image.GetSpacing(),
        remove_small=True, min_component_voxels=10,
        trim_lumen_adjacent=True, lumen_distance_voxels=1,
        erode_core=False, high_hu_threshold=None, low_hu_threshold=None
    )

canonical={r.name:canonical_refine(r) for r in reports}
tpv_rows=[]
for r in reports:
    tpv_rows.append({'vessel':r.name,'raw_tpv_mm3':r.tpv_mm3,
                     'canonical_refined_tpv_mm3':canonical[r.name].refined_tpv_mm3})
tpv_local=pd.DataFrame(tpv_rows)
display(tpv_local)


## 2. Automatically select a coronary-centered CPR rotation

Each frame is treated as an alternate whole-vessel CPR view. We select a central vessel component in each frame, reject border-dominated components, and score the frame using central-vessel continuity plus nearby plaque. Plaque shown in the roadmap must be within 5 mm of that selected 2-D vessel component.

This is **not** a plaque measurement filter. It only decides what appears in the picture.


In [ ]:
DISPLAY_PLAQUE_DISTANCE_MM=5.0
ROADMAP_MARGIN_PX=24

def central_vessel_component(vmask2):
    lab,n=ndi.label(vmask2)
    if n==0: return None
    H,W=vmask2.shape; cy0,cx0=(H-1)/2,(W-1)/2
    best=None; best_score=-np.inf
    for k in range(1,n+1):
        q=(lab==k); area=int(q.sum())
        if area<5: continue
        yy,xx=np.where(q); cy,cx=yy.mean(),xx.mean()
        center_dist=np.hypot((cy-cy0)/H,(cx-cx0)/W)
        touches=((yy==0).any() or (yy==H-1).any() or (xx==0).any() or (xx==W-1).any())
        bbox_span=(yy.max()-yy.min()+1)+(xx.max()-xx.min()+1)
        score=np.log1p(area)+0.004*bbox_span-2.0*center_dist-(1.5 if touches else 0)
        if score>best_score:
            best_score=score; best=q
    return best

def frame_candidate(r, refined, z):
    vessel=central_vessel_component(r.mask[z]==1)
    if vessel is None: return None
    plaque=(refined.refined_mask[z]==2)
    sp_yx=np.array(r.mask_image.GetSpacing(),float)[1::-1]  # y,x
    dist=ndi.distance_transform_edt(~vessel,sampling=sp_yx)
    pdisp=plaque&(dist<=DISPLAY_PLAQUE_DISTANCE_MM)
    yy,xx=np.where(vessel|pdisp)
    if len(yy)==0: return None
    H,W=vessel.shape
    border_fraction=float(np.mean((yy<5)|(yy>=H-5)|(xx<5)|(xx>=W-5)))
    score=float(vessel.sum()) + 8.0*float(pdisp.sum()) - 100.0*border_fraction
    return {'z':int(z),'vessel':vessel,'plaque':pdisp,'score':score,
            'vessel_pixels':int(vessel.sum()),'plaque_pixels':int(pdisp.sum()),
            'border_fraction':border_fraction}

def crop_bounds(mask,shape,margin=24):
    yy,xx=np.where(mask)
    if len(yy)==0: return (0,shape[0],0,shape[1])
    y0=max(0,int(yy.min())-margin); y1=min(shape[0],int(yy.max())+margin+1)
    x0=max(0,int(xx.min())-margin); x1=min(shape[1],int(xx.max())+margin+1)
    return y0,y1,x0,x1

roadmap={}
qc=[]
for r in reports:
    cand=[frame_candidate(r,canonical[r.name],z) for z in range(r.volume.shape[0])]
    cand=[c for c in cand if c is not None]
    cand=sorted(cand,key=lambda c:c['score'],reverse=True)
    best=cand[0]
    y0,y1,x0,x1=crop_bounds(best['vessel']|best['plaque'],best['vessel'].shape,ROADMAP_MARGIN_PX)
    best['crop']=(y0,y1,x0,x1)
    roadmap[r.name]=best
    canonical_vox=int(np.sum(canonical[r.name].refined_mask==2))
    qc.append({'vessel':r.name,'selected_frame':best['z'],'selected_vessel_pixels':best['vessel_pixels'],
               'display_plaque_pixels_in_selected_cpr':best['plaque_pixels'],
               'canonical_3d_plaque_voxels':canonical_vox,
               'note':'Display pixels are a single CPR rotation and are not expected to equal 3D plaque voxels.'})
roadmap_qc=pd.DataFrame(qc); roadmap_qc.to_csv(OUT/'roadmap_qc.csv',index=False)
display(roadmap_qc)


In [ ]:
fig,axs=plt.subplots(3,1,figsize=(15,15))
for ax,r in zip(axs,reports):
    d=roadmap[r.name]; z=d['z']; y0,y1,x0,x1=d['crop']
    img=r.volume[z,y0:y1,x0:x1]
    v=d['vessel'][y0:y1,x0:x1]
    p=d['plaque'][y0:y1,x0:x1]
    ax.imshow(img,cmap='gray',vmin=-200,vmax=800)
    if np.any(v): ax.contour(v,levels=[.5],linewidths=1.2)
    if np.any(p):
        ov=np.ma.masked_where(~p,p)
        ax.imshow(ov,cmap='autumn',alpha=.65,interpolation='nearest')
        ax.contour(p,levels=[.5],linewidths=1.4)
    ax.set_title(f'{r.name} — automatically selected whole-vessel CPR rotation (frame {z})\n'
                 f'orange = plaque near central coronary vessel; thin contour = vessel')
    ax.axis('off')
fig.suptitle('OpenPlaque coronary CPR roadmaps — visualization only; canonical TPV unchanged',fontsize=16)
plt.tight_layout(rect=[0,0,1,.97])
plt.savefig(OUT/'01_coronary_cpr_roadmaps.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 3. RCA PCAT source-CCTA cross-sections

These are the validated artery-centered PCAT views from the frozen RCA centerline/radius model. They remain separate from the CPR plaque views because the two image representations are not spatially registered.


In [ ]:
BASE=ROOT/'PCAT_RCA_10_50'
cp=BASE/'rca_centerline_smoothed_zyx.csv'; rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists(): raise FileNotFoundError('Missing frozen RCA PCAT centerline/radius inputs.')

source_img,ct,_=study.load_series(7); ct=np.asarray(ct,float)
sp_xyz=np.array(source_img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]
cl=pd.read_csv(cp); rad=pd.read_csv(rp)
arc=cl.arc_mm.to_numpy(float); pts_zyx=cl[['z','y','x']].to_numpy(float); pts_mm=pts_zyx*sp_zyx
lumen=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))

aorta_candidates=[ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
                  ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz']
ap=next((p for p in aorta_candidates if p.exists()),None)
if ap is None: raise FileNotFoundError('Missing TotalSegmentator aorta mask.')
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()):
    ai=sitk.Resample(ai,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai).astype(float)

def plane_basis(t):
    t=np.asarray(t,float); t=t/np.linalg.norm(t)
    ref=np.array([1.,0.,0.]) if abs(t[0])<.85 else np.array([0.,1.,0.])
    u=np.cross(t,ref); u=u/np.linalg.norm(u); v=np.cross(t,u)
    return u,v/np.linalg.norm(v)

def sample_plane(target,half=8.,pix=.20):
    i=int(np.argmin(np.abs(arc-target))); i0=max(0,i-2); i1=min(len(arc)-1,i+2)
    tangent=pts_mm[i1]-pts_mm[i0]; tangent=tangent/np.linalg.norm(tangent)
    u,v=plane_basis(tangent); center=pts_mm[i]
    c=np.arange(-half,half+1e-9,pix); U,V=np.meshgrid(c,c,indexing='xy')
    xyz=center[None,None,:]+U[...,None]*u+V[...,None]*v
    vox=(xyz/sp_zyx).reshape(-1,3).T
    img=map_coordinates(ct,vox,order=1,mode='nearest').reshape(U.shape)
    am=map_coordinates(aorta,vox,order=0,mode='nearest').reshape(U.shape)>0.5
    rr=np.sqrt(U**2+V**2); lum=float(lumen[i]); outer=lum+.75; shell_outer=3*outer
    fat=(rr>outer)&(rr<=shell_outer)&(~am)&(img>=-190)&(img<=-30)
    return {'arc':float(arc[i]),'img':img,'fat':fat,'lumen':lum,'outer':outer,
            'shell_outer':shell_outer,'extent':[-half,half,-half,half]}

planes=[sample_plane(x) for x in [10,20,30,40,50]]
fig,axs=plt.subplots(1,5,figsize=(20,4.5))
for ax,d,target in zip(axs,planes,[10,20,30,40,50]):
    ax.imshow(d['img'],cmap='gray',vmin=-200,vmax=800,extent=d['extent'],origin='lower')
    fi=np.ma.masked_where(~d['fat'],d['img'])
    im=ax.imshow(fi,cmap='coolwarm',vmin=-120,vmax=-60,alpha=.8,extent=d['extent'],origin='lower')
    for rr,ls in [(d['lumen'],'-'),(d['outer'],'--'),(d['shell_outer'],':')]:
        ax.add_patch(plt.Circle((0,0),rr,fill=False,linestyle=ls,linewidth=1.7))
    ax.plot(0,0,'+',markersize=9)
    ax.set_title(f'RCA {target} mm\nactual {d["arc"]:.1f} mm')
    ax.set_xlim(-8,8); ax.set_ylim(-8,8); ax.set_aspect('equal'); ax.set_xlabel('mm')
axs[0].set_ylabel('mm')
fig.suptitle('RCA artery-centered PCAT: solid=lumen, dashed=modeled outer wall, dotted=shell edge',fontsize=13)
cb=fig.colorbar(im,ax=axs.ravel().tolist(),shrink=.78,pad=.02); cb.set_label('PCAT attenuation (HU)')
plt.savefig(OUT/'02_rca_pcat_cross_sections.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
lp=ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_canonical_longitudinal_v2.csv'
longdf=pd.read_csv(lp)
x=(longdf.arc_start_mm.to_numpy(float)+longdf.arc_end_mm.to_numpy(float))/2
y=longdf.mean_hu.to_numpy(float)
fig,ax=plt.subplots(figsize=(12,3.2))
sc=ax.scatter(x,np.zeros_like(x),c=y,cmap='coolwarm',vmin=-110,vmax=-75,s=260,marker='s')
ax.set_xlim(10,50); ax.set_yticks([]); ax.set_xlabel('Distance from RCA ostium (mm)')
ax.set_title('RCA longitudinal PCAT attenuation ribbon')
cb=fig.colorbar(sc,ax=ax,pad=.02); cb.set_label('Mean HU per 1-mm segment')
plt.tight_layout(); plt.savefig(OUT/'03_rca_longitudinal_pcat_ribbon.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 4. One-page summary

The plaque roadmaps and PCAT views are intentionally presented as separate panels because they come from different representations.


In [ ]:
tpv=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'tpv_metrics_by_vessel_v2.csv')
pcat=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_canonical_primary_v2.csv')
ps=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_circular_sensitivity_v2.csv')
total=tpv[tpv.vessel=='TOTAL'].iloc[0]; primary=pcat.iloc[0]
cmin=float(ps.pcat_mean_hu.min()); cmax=float(ps.pcat_mean_hu.max())
directional=np.nan
dp=ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv'
if dp.exists():
    d=pd.read_csv(dp)
    directional=float(d.iloc[0]['directional_pcat_mean_hu']) if 'directional_pcat_mean_hu' in d.columns else np.nan
fmin=min(cmin,directional) if np.isfinite(directional) else cmin
fmax=max(cmax,directional) if np.isfinite(directional) else cmax

fig=plt.figure(figsize=(13,8))
gs=fig.add_gridspec(2,2,height_ratios=[1,1.1])
ax1=fig.add_subplot(gs[0,0]); vv=tpv[tpv.vessel!='TOTAL']
ax1.bar(vv.vessel,vv.canonical_refined_tpv_mm3); ax1.set_ylabel('Refined TPV (mm³)'); ax1.set_title('Plaque volume by artery')
ax2=fig.add_subplot(gs[0,1]); ax2.axis('off')
txt=(f'Total refined TPV: {total.canonical_refined_tpv_mm3:.0f} mm³\n'
     f'Raw TPV: {total.raw_tpv_mm3:.0f} mm³\n'
     f'TPV sensitivity: {total.sensitivity_min_mm3:.0f}–{total.sensitivity_max_mm3:.0f} mm³\n\n'
     f'RCA 10–50 mm PCAT: {primary.pcat_mean_hu:.2f} HU\n'
     f'Circular-margin range: {cmin:.2f} to {cmax:.2f} HU\n'
     f'Full tested geometry range: {fmin:.2f} to {fmax:.2f} HU\n\n'
     'Plaque CPR and PCAT source-volume views are not spatially co-registered.')
ax2.text(.02,.96,txt,va='top',fontsize=12,
         bbox=dict(boxstyle='round',facecolor='white',alpha=.9))
ax3=fig.add_subplot(gs[1,:])
sc=ax3.scatter(x,np.zeros_like(x),c=y,cmap='coolwarm',vmin=-110,vmax=-75,s=340,marker='s')
ax3.set_xlim(10,50); ax3.set_yticks([]); ax3.set_xlabel('Distance from RCA ostium (mm)')
ax3.set_title('RCA longitudinal PCAT attenuation')
cb=fig.colorbar(sc,ax=ax3,pad=.02); cb.set_label('HU')
fig.suptitle('OpenPlaque plaque + PCAT summary',fontsize=17)
plt.tight_layout(rect=[0,0,1,.96]); plt.savefig(OUT/'04_summary_dashboard.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
summary=pd.DataFrame([{
    'total_refined_tpv_mm3':float(total.canonical_refined_tpv_mm3),
    'total_raw_tpv_mm3':float(total.raw_tpv_mm3),
    'tpv_sensitivity_min_mm3':float(total.sensitivity_min_mm3),
    'tpv_sensitivity_max_mm3':float(total.sensitivity_max_mm3),
    'rca_pcat_mean_hu':float(primary.pcat_mean_hu),
    'pcat_circular_min_hu':cmin,'pcat_circular_max_hu':cmax,
    'pcat_full_tested_min_hu':fmin,'pcat_full_tested_max_hu':fmax
}])
summary.to_csv(OUT/'roadmap_report_summary.csv',index=False)

# Build a simple self-contained HTML report.
def img64(p):
    return base64.b64encode(Path(p).read_bytes()).decode('ascii')
imgs=['01_coronary_cpr_roadmaps.png','02_rca_pcat_cross_sections.png',
      '03_rca_longitudinal_pcat_ribbon.png','04_summary_dashboard.png']
html=['<html><head><meta charset="utf-8"><title>OpenPlaque Coronary Roadmap</title></head><body>',
      '<h1>OpenPlaque Coronary CPR Roadmap + RCA PCAT</h1>',
      '<p><b>Research use only.</b> Plaque CPR views and PCAT source-volume views are not spatially co-registered. PCAT attenuation is a surrogate related to perivascular inflammation, not a direct inflammation measurement.</p>']
for f in imgs:
    html += [f'<h2>{f}</h2>',f'<img style="max-width:100%;" src="data:image/png;base64,{img64(OUT/f)}">']
html += ['</body></html>']
(OUT/'OPENPLAQUE_CORONARY_ROADMAP_REPORT.html').write_text('\n'.join(html),encoding='utf-8')

zip_path=OUT/'OPENPLAQUE_CORONARY_ROADMAP_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.name==zip_path.name: continue
        if p.suffix.lower() in ['.png','.csv','.html']:
            z.write(p,arcname=p.name)
print('Report:',OUT/'OPENPLAQUE_CORONARY_ROADMAP_REPORT.html')
print('ZIP:',zip_path)
